# 10 · Accelerate — M×N GPU

native `accelerate launch --multi_gpu --num_machines M --machine_rank R`은 노드별 dispatch를 요구해 Databricks 단일 driver 노트북에서 직접 띄울 수 없다. 실용적인 패턴은 **TorchDistributor를 dispatcher로** 사용해 worker 노드에 child 프로세스를 분산 spawn하고, child 안에서 raw `dist.*` 대신 **HuggingFace `Accelerator()` API**를 사용하는 것이다. TorchDistributor가 세팅한 `RANK`/`WORLD_SIZE`/`MASTER_ADDR`/`MASTER_PORT`를 `Accelerator()`가 자동으로 인식한다.

> 1×1은 [`08-launch_accelerator_1x1.ipynb`](08-launch_accelerator_1x1.ipynb), 1×N은 [`09-launch_accelerator_1xN.ipynb`](09-launch_accelerator_1xN.ipynb).

**클러스터**: Multi-node Classic GPU, DBR 17.3 LTS ML, Workers `g5.12xlarge` × M.

In [ ]:
%run ./00-setup

## SCRIPT_DIR

In [ ]:
import os
import sys

NOTEBOOK_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
SCRIPT_DIR = "/Workspace" + os.path.dirname(NOTEBOOK_PATH)
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)
print(f"SCRIPT_DIR={SCRIPT_DIR}")

## 학습 함수 (Accelerator API)

inline 정의해 by-value pickling. child 내부에서 sys.path 보강 후 `model.TwoTowerMLP` import.

In [ ]:
def train_fn_acc(
    run_id,
    db_host,
    db_token,
    data_dir,
    ckpt_path,
    n_users,
    n_items,
    emb_dim,
    tower_hidden,
    batch_size,
    num_epochs,
    max_steps_per_epoch,
    patience,
    min_delta,
    topology,
    script_dir,
):
    import os
    import sys

    if script_dir and script_dir not in sys.path:
        sys.path.insert(0, script_dir)

    import mlflow
    import pyarrow.parquet as pq
    import torch
    import torch.nn as nn
    from accelerate import Accelerator
    from torch.utils.data import DataLoader, TensorDataset
    from torch.utils.data.distributed import DistributedSampler

    from model import EarlyStopping, TwoTowerMLP

    os.environ["DATABRICKS_HOST"] = db_host
    os.environ["DATABRICKS_TOKEN"] = db_token

    accelerator = Accelerator()
    device = accelerator.device
    global_rank = accelerator.process_index
    world_size = accelerator.num_processes

    def load_split(split):
        split_dir = os.path.join(data_dir, split)
        files = sorted(
            os.path.join(split_dir, f) for f in os.listdir(split_dir) if f.endswith(".parquet")
        )
        table = pq.read_table(files, columns=["user_id", "item_id", "label"])
        return TensorDataset(
            torch.from_numpy(table.column("user_id").to_numpy()),
            torch.from_numpy(table.column("item_id").to_numpy()),
            torch.from_numpy(table.column("label").to_numpy()),
        )

    train_dataset = load_split("train")
    val_dataset = load_split("val")
    train_sampler = DistributedSampler(train_dataset, num_replicas=world_size, rank=global_rank, drop_last=True)
    val_sampler = DistributedSampler(val_dataset, num_replicas=world_size, rank=global_rank, shuffle=False, drop_last=False)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, sampler=val_sampler, num_workers=2, pin_memory=True)

    model = TwoTowerMLP(n_users, n_items, emb_dim, tower_hidden)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    model, optimizer, train_loader, val_loader = accelerator.prepare(
        model, optimizer, train_loader, val_loader
    )
    early_stop = EarlyStopping(patience=patience, min_delta=min_delta)

    if accelerator.is_main_process:
        mlflow.start_run(run_id=run_id, log_system_metrics=True)
        mlflow.log_params({
            "topology": topology,
            "world_size": world_size,
            "n_users": n_users,
            "n_items": n_items,
            "emb_dim": emb_dim,
            "batch_size": batch_size,
            "num_epochs": num_epochs,
            "max_steps_per_epoch": max_steps_per_epoch,
            "patience": patience,
            "min_delta": min_delta,
            "code_organization": "02-script-based",
            "launcher": "accelerate-via-torchdistributor",
        })

    global_step = 0
    for epoch in range(num_epochs):
        train_sampler.set_epoch(epoch)
        model.train()
        for step, (u, i, y) in enumerate(train_loader):
            if step >= max_steps_per_epoch:
                break
            optimizer.zero_grad(set_to_none=True)
            logits = model(u, i)
            loss = loss_fn(logits, y)
            accelerator.backward(loss)
            optimizer.step()
            if accelerator.is_main_process and step % 20 == 0:
                mlflow.log_metric("train/loss", loss.item(), step=global_step)
            global_step += 1

        model.eval()
        local_loss_sum = torch.zeros(1, device=device)
        local_n = torch.zeros(1, device=device)
        with torch.no_grad():
            for u, i, y in val_loader:
                logits = model(u, i)
                loss = loss_fn(logits, y)
                local_loss_sum += loss * y.size(0)
                local_n += y.size(0)
        local_loss_sum = accelerator.reduce(local_loss_sum, reduction="sum")
        local_n = accelerator.reduce(local_n, reduction="sum")
        val_loss = (local_loss_sum / local_n).item()

        if accelerator.is_main_process:
            mlflow.log_metric("val/loss", val_loss, step=epoch)
            print(
                f"[rank=0] epoch={epoch:2d} val_loss={val_loss:.6f} "
                f"(best={early_stop.best:.6f} counter={early_stop.counter})"
            )

        if early_stop.step(val_loss):
            if accelerator.is_main_process:
                print(f"early stop at epoch {epoch} (best val_loss={early_stop.best:.6f})")
                mlflow.log_metric("early_stop_epoch", epoch)
            break

    accelerator.wait_for_everyone()
    if accelerator.is_main_process:
        unwrapped = accelerator.unwrap_model(model)
        torch.save({"model": unwrapped.state_dict()}, ckpt_path)
        mlflow.log_param("ckpt_path", ckpt_path)
        mlflow.end_run()
        print(f"saved {ckpt_path}")
    return "ok" 

## M×N GPU

In [ ]:
import mlflow
from pyspark.ml.torch.distributor import TorchDistributor

NUM_NODES = 2          # M
NUM_GPUS_PER_NODE = 4  # N

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="acc-MxN", log_system_metrics=True) as run:
    run_id = run.info.run_id

TorchDistributor(
    num_processes=NUM_NODES * NUM_GPUS_PER_NODE,
    local_mode=False,
    use_gpu=True,
).run(
    train_fn_acc,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_path=os.path.join(CKPT_DIR, "acc-MxN.pt"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    topology="MxN",
    script_dir=SCRIPT_DIR,
)